# 03 — EHR ETL Pipeline

Normalize raw EHR CSV exports to standard vocabularies using PyHealth's CrossMap,
then write parquet files ready for dataset loading.

This pattern handles real-world hospital data that uses ICD-9, NDC codes, etc.
and normalizes them to ICD-10 and ATC for use with PyHealth models.

In [ ]:
from pathlib import Path
from pyhealth_enterprise.config import settings
from pyhealth_enterprise.pipelines.ehr_etl import EHRETL

source_dir = settings.SYNTHETIC_DATA_PATH
output_dir = settings.PROJECT_ROOT / "data" / "processed" / "normalized"

etl = EHRETL(source_dir=source_dir, output_dir=output_dir)
etl.run()

In [ ]:
# Verify outputs
import pandas as pd

patients = pd.read_parquet(output_dir / "patients.parquet")
diagnoses = pd.read_parquet(output_dir / "diagnoses.parquet")

print(f"Patients: {len(patients)}")
print(f"Diagnoses: {len(diagnoses)}")
print("\nSample diagnosis normalization (ICD9 -> ICD10):")
diagnoses[["icd_code", "icd10_code"]].head(10)

In [ ]:
# Explore CrossMap directly
from pyhealth.medcode import CrossMap

icd9_to_icd10 = CrossMap("ICD9CM", "ICD10CM")

test_codes = ["428.0", "250.00", "401.9", "410.9"]
for code in test_codes:
    mapped = icd9_to_icd10.map(code)
    print(f"  ICD9 {code:10s} -> ICD10 {mapped}")

In [ ]:
# Explore ATC code lookup
from pyhealth.medcode import ATC

example_atc_codes = ["A02BC01", "B01AC06", "C10AA01"]
for code in example_atc_codes:
    try:
        drug = ATC(code)
        print(f"  {code}: {drug.name}")
    except Exception as e:
        print(f"  {code}: {e}")